In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import jsonlines
from pathlib import Path
import plotly.express as px
import numpy as np
from reader import load_messages, deduplicate_messages, inspect_keys, get_value, normalize_model_name

Download data with a command like:

```sh
pixi shell -e llmaven
llmaven infra extract --from 2026-01-01 --to 2026-03-31 --out jan-feb-march-2026.zip -e .env
unzip jan-feb-march-2026.zip
cat litellm*.jsonl > jan-feb-mar-2026.jsonl
```

In [ ]:
# path = Path("session_tests/litellm_spend_logs_2026-03-27.jsonl")
path = Path("jan-feb-mar-2026.jsonl")
with jsonlines.open(path) as reader:
    data = [record for record in reader]
print(f"Loaded {len(data)} records")

In [ ]:
# check the unique set of keys in the data. Map each record to its set of keys, then take the unique sets
keys = set(tuple(record.keys()) for record in data)
print(len(keys))

inspect_keys(data, ["end_user"])

data_filt = list(filter(lambda record: get_value(record, ["metadata","user_api_key_alias"]) not in {"safemind-gh-repo-anant", "patient-psi"}, data))
print(f"Filtered data has {len(data_filt)} records vs {len(data)} original records")

In [ ]:

# histogram of records by normalized model, grouping rare models as "other"
model_counts = pd.Series(
    [normalize_model_name(record.get("model", "")) for record in data_filt if "model" in record]
).value_counts()

top_models = model_counts[model_counts > 20]
other_count = model_counts[model_counts <= 20].sum()
if other_count > 0:
    top_models["other"] = other_count

fig = px.bar(top_models, title="Distribution of Records by Model (< 20 records grouped as 'other')",
             labels={"index": "Model", "value": "Count"})
fig.show()

In [ ]:
# box plot of prompt tokens, completion tokens, and total tokens
df = pd.DataFrame(data_filt)
cols= ["prompt_tokens", "completion_tokens"]
# cols= ["completion_tokens"]
fig = px.box(df, y=cols,
             title="Distribution of Tokens",
             labels={"value": "Number of Tokens", "variable": "Token Type"})
fig.show()

In [ ]:
dfm = deduplicate_messages(load_messages(path))
dfm.columns

In [ ]:
# example of how to reconstruct the user messages in a session, showing the tool calls as "tool: tool_name" and the text messages as "text: text"
# df_user = dfm[(dfm["direction"] == "input") & (dfm["session_id"] != '')]# & (dfm["role"] == "user")]
# df_user["display"] = df_user.apply(lambda row: f"tool: {row['tool_name']}" if pd.notnull(row['tool_name']) else f"text: {row['text']}", axis=1)   
# df_user[ df_user["session_id"] == "32bd207c-970f-47dd-a25b-c9d8a18c4731" ][["request_id", "start_time", "msg_idx", "role", "type", "display"]]

In [ ]:
safemind_keys = {"safemind-gh-repo-anant", "patient-psi"}
# Histogram: number of (user) messages per session
msgs_per_session = (
    dfm[(dfm["direction"] == "input") & (dfm["role"] == "user") & (dfm["model"].str.contains("claude")) & (~dfm["user_api_key_alias"].isin(safemind_keys))]
    .groupby(["device_id", "account_uuid", "session_id"])["msg_idx"]
    .nunique()
    .reset_index(name="n_messages")
)
max_pow = int(np.ceil(np.log2(max(msgs_per_session["n_messages"]))))
bin_edges = [0.5] + [2**i + 0.5 for i in range(max_pow + 1)]
bin_labels = [f"≤{2**i}" for i in range(max_pow + 1)]

counts, _ = np.histogram(msgs_per_session["n_messages"], bins=bin_edges)

fig = px.bar(
    x=bin_labels,
    y=counts,
    title="Distribution of Number of Messages in Proxy Server Requests",
    labels={"x": "Number of Messages", "y": "Count"},
)
fig.show()

In [ ]:
# Sample 10 messages from sessions with n_messages=1
single_msg_sessions = msgs_per_session[msgs_per_session["n_messages"] == 1]
sample_sessions = single_msg_sessions.sample(10, random_state=42)["session_id"]

sample_msgs = dfm[
    (dfm["session_id"].isin(sample_sessions))
    & (dfm["direction"] == "input")
    & (dfm["role"] == "user")
    & (dfm["type"] == "text")
][["session_id", "text"]].drop_duplicates()

for _, row in sample_msgs.iterrows():
    print(f"[{row['session_id']}]\n{row['text'][:120]}\n{'-'*80}")